# Install Package

In [ ]:
!pip install textattack
!pip install tensorflow_hub -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install numpy==1.25.0
!pip install tensorflow_text

In [ ]:
import nltk
nltk.set_proxy('http://localhost:7890')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

# One Attack

In [ ]:
import textattack
import transformers

# Load model, tokenizer, and model_wrapper
model = transformers.AutoModelForSequenceClassification.from_pretrained("textattack/bert-base-uncased-sst2")
tokenizer = transformers.AutoTokenizer.from_pretrained("textattack/bert-base-uncased-sst2")
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

# Construct our four components for `Attack`
from textattack.constraints.pre_transformation import RepeatModification, StopwordModification
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.transformations import WordSwapEmbedding
from textattack.search_methods import GreedyWordSwapWIR

goal_function = textattack.goal_functions.UntargetedClassification(model_wrapper)
constraints = [
    RepeatModification(),
    StopwordModification(),
    WordEmbeddingDistance(min_cos_sim=0.9)
]
transformation = WordSwapEmbedding(max_candidates=50)
search_method = GreedyWordSwapWIR(wir_method="delete")

# Construct the actual attack
attack = textattack.Attack(goal_function, constraints, transformation, search_method)

input_text = "I really enjoyed the new movie that came out last month."
label = 1 #Positive
attack_result = attack.attack(input_text, label)

# Train

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score
import numpy as np
import tqdm
torch.cuda.set_device(2)

def fine_tune_bert(train_file, test_file, model_path=None, new_model_path=None, num_epochs=3, batch_size=4, learning_rate=5e-5, label_map={'benign': 0, 'illicit': 1}):
    # 加载或初始化分词器和模型
    if model_path:
        tokenizer = BertTokenizer.from_pretrained(model_path)
        model = BertForSequenceClassification.from_pretrained(model_path, num_labels=2)
    else:
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    
    # 读取数据集
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    
    
    train_df['label'] = train_df['label'].map(label_map)
    test_df['label'] = test_df['label'].map(label_map)
    
    # 分割数据集
    train_texts = train_df['text'].tolist()
    train_labels = train_df['label'].tolist()
    test_texts = test_df['text'].tolist()
    test_labels = test_df['label'].tolist()
    
    # 创建数据集
    class TextDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=512):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_len = max_len
        
        def __len__(self):
            return len(self.texts)
        
        def __getitem__(self, idx):
            text = self.texts[idx]
            label = self.labels[idx]
            encoding = self.tokenizer.encode_plus(
                text,
                add_special_tokens=True,
                max_length=self.max_len,
                return_attention_mask=True,
                return_tensors='pt',
                padding='max_length',
                truncation=True
            )
            return {
                'input_ids': encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten(),
                'labels': torch.tensor(label, dtype=torch.long)
            }
    
    train_dataset = TextDataset(train_texts, train_labels, tokenizer)
    test_dataset = TextDataset(test_texts, test_labels, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # 设置优化器和调度器
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    
    # 训练模型
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm.tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=True)
        for batch in progress_bar:
            b_input_ids = batch['input_ids'].to(device)
            b_attn_mask = batch['attention_mask'].to(device)
            b_labels = batch['labels'].to(device)
            model.zero_grad()
            outputs = model(input_ids=b_input_ids, attention_mask=b_attn_mask, labels=b_labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            scheduler.step()
            progress_bar.set_postfix(loss=loss.item())
        print(f'Epoch {epoch+1}, Train Loss: {total_loss / len(train_loader):.4f}')
    
    # 评估模型
    def evaluate_model(model, data_loader, device):
        model.eval()
        total_acc = 0
        with torch.no_grad():
            for batch in data_loader:
                b_input_ids = batch['input_ids'].to(device)
                b_attn_mask = batch['attention_mask'].to(device)
                b_labels = batch['labels'].to(device)
                outputs = model(input_ids=b_input_ids, attention_mask=b_attn_mask, labels=b_labels)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=1).flatten()
                acc = accuracy_score(b_labels.cpu(), preds.cpu())
                total_acc += acc
        return total_acc / len(data_loader)
    
    test_acc = evaluate_model(model, test_loader, device)
    print(f'Test Accuracy: {test_acc:.4f}')
    # 保存模型
    
    model_path = new_model_path
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)




In [ ]:
# 使用函数进行微调
fine_tune_bert('/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/train.csv', '/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/test.csv', num_epochs=10, batch_size=4, learning_rate=5e-5, label_map = {'benign': 0, 'toxic': 1}, model_path='/data/rhhuang/models/bert/bert-base-uncased', new_model_path='/data/rhhuang/models/bert/bert-base-uncased-toxic')

In [ ]:
# 使用函数进行微调
fine_tune_bert('/data/ningyuanhe/EvasionFromICL/data/sentiment/train.csv', '/data/ningyuanhe/EvasionFromICL/data/sentiment/test.csv', num_epochs=10, batch_size=4, learning_rate=5e-5, label_map = {'negative': 0, 'positive': 1}, model_path='/data/rhhuang/models/bert/bert-base-uncased', new_model_path='/data/rhhuang/models/bert/bert-base-uncased-sst2')

In [ ]:
# 使用函数进行微调
fine_tune_bert('/data/ningyuanhe/EvasionFromICL/data/illicit_promotion//train.csv', '/data/ningyuanhe/EvasionFromICL/data/illicit_promotion//test.csv', num_epochs=10, batch_size=4, learning_rate=5e-5, label_map = {'benign': 0, 'illicit': 1}, model_path='/data/rhhuang/models/bert/bert-base-multilingual-cased', new_model_path='/data/rhhuang/models/bert/bert-base-multilingual-cased-illicit')

# Test BERT Args

In [ ]:
import torch
torch.cuda.set_device(2)
# 定义参数
batch_sizes = [16, 32]
learning_rates = [5e-5, 3e-5, 2e-5]
num_epochs_list = [2, 3, 4]
train_file = '/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/train.csv'
test_file = '/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/test.csv'
base_model_path = '/data/rhhuang/models/bert/bert-base-uncased'

# 网格搜索
for batch_size in batch_sizes:
    for learning_rate in learning_rates:
        for num_epochs in num_epochs_list:
            # 构建保存路径
            new_model_path = f'/data/rhhuang/models/bert/bert-base-uncased-toxic-batch{batch_size}-lr{learning_rate}-epochs{num_epochs}'
            
            # 调用微调函数
            print(f"开始训练：批量大小={batch_size}, 学习率={learning_rate}, epoch={num_epochs}")
            fine_tune_bert(
                train_file=train_file,
                test_file=test_file,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                label_map={'benign': 0, 'toxic': 1},
                model_path=base_model_path,
                new_model_path=new_model_path
            )
            print(f"完成训练并保存至：{new_model_path}")


In [ ]:
import torch
torch.cuda.set_device(1)
# 定义参数
batch_sizes = [16, 32]
learning_rates = [5e-5, 3e-5, 2e-5]
num_epochs_list = [2, 3, 4]
train_file = '/data/ningyuanhe/EvasionFromICL/data/sentiment/train.csv'
test_file = '/data/ningyuanhe/EvasionFromICL/data/sentiment/test.csv'
base_model_path = '/data/rhhuang/models/bert/bert-base-uncased'

# 网格搜索
for batch_size in batch_sizes:
    for learning_rate in learning_rates:
        for num_epochs in num_epochs_list:
            # 构建保存路径
            new_model_path = f'/data/rhhuang/models/bert/bert-base-uncased-sentiment-batch{batch_size}-lr{learning_rate}-epochs{num_epochs}'
            
            # 调用微调函数
            print(f"开始训练：批量大小={batch_size}, 学习率={learning_rate}, epoch={num_epochs}")
            fine_tune_bert(
                train_file=train_file,
                test_file=test_file,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                label_map={'negative': 0, 'positive': 1},
                model_path=base_model_path,
                new_model_path=new_model_path
            )
            print(f"完成训练并保存至：{new_model_path}")


In [ ]:
import torch
torch.cuda.set_device(0)
# 定义参数
batch_sizes = [16, 32]
learning_rates = [5e-5, 3e-5, 2e-5]
num_epochs_list = [2, 3, 4]
train_file = '/data/ningyuanhe/EvasionFromICL/data/illicit_promotion//train.csv'
test_file = '/data/ningyuanhe/EvasionFromICL/data/illicit_promotion//test.csv'
base_model_path = '/data/rhhuang/models/bert/bert-base-multilingual-cased'

# 网格搜索
for batch_size in batch_sizes:
    for learning_rate in learning_rates:
        for num_epochs in num_epochs_list:
            # 构建保存路径
            new_model_path = f'/data/rhhuang/models/bert/bert-base-multilingual-cased-illicit-batch{batch_size}-lr{learning_rate}-epochs{num_epochs}'
            
            # 调用微调函数
            print(f"开始训练：批量大小={batch_size}, 学习率={learning_rate}, epoch={num_epochs}")
            fine_tune_bert(
                train_file=train_file,
                test_file=test_file,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                label_map={'benign': 0, 'illicit': 1},
                model_path=base_model_path,
                new_model_path=new_model_path
            )
            print(f"完成训练并保存至：{new_model_path}")


# Evaluate

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
torch.cuda.set_device(2)
def predict_with_bert(model_path, test_file, batch_size=4, max_len=512, pos_label='benign', nag_label='illicit'):
    # 检测是否有可用的GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 加载分词器和模型
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path, num_labels=2).to(device)
    model.eval()  # 设置为评估模式
    
    # 读取测试数据集
    test_df = pd.read_csv(test_file)
    test_texts = test_df['text'].tolist()
    test_labels = test_df['label'].tolist()  # 保留真实标签以计算指标
    
    # 创建数据集
    class TextDataset(Dataset):
        def __init__(self, texts, tokenizer, max_len=512):
            self.texts = texts
            self.tokenizer = tokenizer
            self.max_len = max_len
        
        def __len__(self):
            return len(self.texts)
        
        def __getitem__(self, idx):
            text = self.texts[idx]
            encoding = self.tokenizer.encode_plus(
                text,
                add_special_tokens=True,
                max_length=self.max_len,
                return_attention_mask=True,
                return_tensors='pt',
                padding='max_length',
                truncation=True
            )
            return {
                'input_ids': encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten()
            }
    
    test_dataset = TextDataset(test_texts, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # 进行预测
    predictions = []
    progress_bar = tqdm(test_loader, desc='Predicting', leave=True)
    with torch.no_grad():
        for batch in progress_bar:
            b_input_ids = batch['input_ids'].to(device)
            b_attn_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=b_input_ids, attention_mask=b_attn_mask)
            logits = outputs.logits
            pred = torch.argmax(logits, dim=1).flatten()
            predictions.extend(pred.cpu().numpy())
            # 更新进度条描述
            progress_bar.set_postfix({'step': len(predictions)})
    
    # 将预测结果转换回标签
    pred_labels = [pos_label if pred == 0 else nag_label for pred in predictions]
    
    # 计算指标
    accuracy = accuracy_score(test_labels, [pos_label if pred == 0 else nag_label for pred in predictions])
    precision = precision_score(test_labels, [pos_label if pred == 0 else nag_label for pred in predictions], pos_label=nag_label)
    recall = recall_score(test_labels, [pos_label if pred == 0 else nag_label for pred in predictions], pos_label=nag_label)
    f1 = f1_score(test_labels, [pos_label if pred == 0 else nag_label for pred in predictions], pos_label=nag_label)
    
    return pred_labels, accuracy, precision, recall, f1



In [ ]:
# 使用函数进行预测
model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
test_file = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion/test.csv" 
predictions, accuracy, precision, recall, f1 = predict_with_bert(model_path, test_file,pos_label='benign', nag_label='illicit')
print(f"Predictions: {predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# 使用函数进行预测
model_path = "/data/rhhuang/models/bert/bert-base-uncased-toxic"
test_file = "/data/ningyuanhe/EvasionFromICL/data/toxic_text/data/test.csv" 
predictions, accuracy, precision, recall, f1 = predict_with_bert(model_path, test_file, pos_label='benign', nag_label='toxic')
print(f"Predictions: {predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# 使用函数进行预测
model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
test_file = "/data/ningyuanhe/EvasionFromICL/data/sentiment/test.csv" 
predictions, accuracy, precision, recall, f1 = predict_with_bert(model_path, test_file, pos_label='negative', nag_label='positive')
print(f"Predictions: {predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Define Attack

In [2]:
import textattack
import transformers
import os
import csv
import pandas as pd


def run_adv_attack_experiment(
        attack,
        attack_name,
        dataset_path="/data/rhhuang/notebooks/Adv4ICL/data/sentiment/",
        random_seed=42,
        result_base_dir="/data/rhhuang/notebooks/AdvTradition",
        query_budget=None,
        label2values={'positive': 1, 'negative': 0},
        pos_label='negative',
        max_num_words=1
    ):
    test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
    test_df = test_df[test_df['label'] == pos_label]
    test_df['label_value'] = test_df['label'].map(label2values)
    
    # 创建 text 到 label 的映射
    text_label_map = dict(zip(test_df['text'], test_df['label']))

    # 创建 text 到 idx 的映射
    text_idx_map = dict(zip(test_df['text'], test_df['idx']))

    # 提取所需数据
    test_data = [(row['text'], row['label_value']) for _, row in test_df.iterrows()]
    # 创建TextAttack的数据集
    eval_dataset = textattack.datasets.Dataset(test_data)

    os.makedirs(result_base_dir, exist_ok=True)
    # Attack 20 samples with CSV logging and checkpoint saved every 5 interval
    attack_args = textattack.AttackArgs(
        num_examples=-1,
        log_to_csv=os.path.join(result_base_dir, "log.csv"),
        checkpoint_interval=5,
        disable_stdout=True,
        query_budget=query_budget,
        random_seed=random_seed,
    )

    attacker = textattack.Attacker(attack, eval_dataset, attack_args)
    result = attacker.attack_dataset()
    
    output_file = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_resultsV2.csv")
    output_file_success = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_results_successV2.csv")
    output_file_fail = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_results_failV2.csv")
    output_file_skip = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_results_skipV2.csv")
    output_file_extra = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_extra_resultsV2.csv")
    
    # 定义文件头
    header = ["original_text", "perturbed_text", "label"]
    extra_header = ["idx", "text", "label_value", "label"]

    # 读取并保存结果
    with open(output_file_success, mode="w", newline='', encoding="utf-8") as file_success, \
         open(output_file_skip, mode="w", newline='', encoding="utf-8") as file_skip, \
         open(output_file_fail, mode="w", newline='', encoding="utf-8") as file_fail, \
         open(output_file, mode="w", newline='', encoding="utf-8") as file, \
         open(output_file_extra, mode="w", newline='', encoding="utf-8") as file_extra:

        writer = csv.writer(file)
        writer.writerow(header)

        writer_success = csv.writer(file_success)
        writer_success.writerow(header)

        writer_fail = csv.writer(file_fail)
        writer_fail.writerow(header)

        writer_skip = csv.writer(file_skip)
        writer_skip.writerow(header)

        extra_writer = csv.writer(file_extra)
        extra_writer.writerow(extra_header)

        for attack_result in result:
            original_text = attack_result.original_text()
            perturbed_text = attack_result.perturbed_text()
            label = text_label_map[original_text]
            label_value = label2values[label]
            
            if isinstance(attack_result, textattack.attack_results.SuccessfulAttackResult):
                writer_success.writerow([original_text, perturbed_text, label])
            elif isinstance(attack_result, textattack.attack_results.FailedAttackResult):
                writer_fail.writerow([original_text, perturbed_text, label])
            elif isinstance(attack_result, textattack.attack_results.SkippedAttackResult):
                writer_skip.writerow([original_text, perturbed_text, label])
            else:
                raise Exception("Unknown Type!")
            
            writer.writerow([original_text, perturbed_text, label])
            idx = text_idx_map[original_text]
            extra_writer.writerow([idx, perturbed_text, label_value, label])

    print(f"攻击结果已保存到 {output_file}")
    print(f"附加结果已保存到 {output_file_extra}")


# Define My attack

In [1]:
"""

TextFooler (Is BERT Really Robust?)
===================================================
A Strong Baseline for Natural Language Attack on Text Classification and Entailment)

"""
import textattack
from textattack import Attack
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.pre_transformation import (
    InputColumnModification,
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapEmbedding




class AddMaxWordTextFoolerJin2019(textattack.attack_recipes.TextFoolerJin2019):
    """Jin, D., Jin, Z., Zhou, J.T., & Szolovits, P. (2019).

    Is BERT Really Robust? Natural Language Attack on Text
    Classification and Entailment.

    https://arxiv.org/abs/1907.11932
    """

    @staticmethod
    def build(model_wrapper, max_num_words = None, max_percent = None):
        #
        # Swap words with their 50 closest embedding nearest-neighbors.
        # Embedding: Counter-fitted PARAGRAM-SL999 vectors.
        #
        transformation = WordSwapEmbedding(max_candidates=50)
        #
        # Don't modify the same word twice or the stopwords defined
        # in the TextFooler public implementation.
        #
        # fmt: off
        stopwords = set(
            ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
        )
        # fmt: on
        constraints = [RepeatModification(), StopwordModification(stopwords=stopwords)]
        #
        # During entailment, we should only edit the hypothesis - keep the premise
        # the same.
        #
        input_column_modification = InputColumnModification(
            ["premise", "hypothesis"], {"premise"}
        )
        constraints.append(input_column_modification)
        # Minimum word embedding cosine similarity of 0.5.
        # (The paper claims 0.7, but analysis of the released code and some empirical
        # results show that it's 0.5.)
        #
        constraints.append(WordEmbeddingDistance(min_cos_sim=0.5))
        #
        # Only replace words with the same part of speech (or nouns with verbs)
        #
        constraints.append(PartOfSpeech(allow_verb_noun_swap=True))
        #
        # Universal Sentence Encoder with a minimum angular similarity of ε = 0.5.
        #
        # In the TextFooler code, they forget to divide the angle between the two
        # embeddings by pi. So if the original threshold was that 1 - sim >= 0.5, the
        # new threshold is 1 - (0.5) / pi = 0.840845057
        #
        use_constraint = UniversalSentenceEncoder(
            threshold=0.840845057,
            metric="angular",
            compare_against_original=False,
            window_size=15,
            skip_text_shorter_than_window=True,
        )
        constraints.append(use_constraint)

        # add max_words_perturbed
        if max_num_words or max_percent:
            constraints.append(textattack.constraints.overlap.MaxWordsPerturbed(max_num_words=max_num_words, max_percent=max_percent))
        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # Greedily swap words with "Word Importance Ranking".
        #
        search_method = GreedyWordSwapWIR(wir_method="delete")

        return Attack(goal_function, constraints, transformation, search_method)


2025-01-09 12:49:25.831056: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-09 12:49:25.845697: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736426965.865548 1092723 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736426965.871589 1092723 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-09 12:49:25.892216: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [8]:
"""

TextBugger
===============

(TextBugger: Generating Adversarial Text Against Real-world Applications)

"""
import textattack
from textattack import Attack
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import (
    CompositeTransformation,
    WordSwapEmbedding,
    WordSwapHomoglyphSwap,
    WordSwapNeighboringCharacterSwap,
    WordSwapRandomCharacterDeletion,
    WordSwapRandomCharacterInsertion,
)




class AddMaxWordTextBuggerLi2018(textattack.attack_recipes.TextBuggerLi2018):
    """Li, J., Ji, S., Du, T., Li, B., and Wang, T. (2018).

    TextBugger: Generating Adversarial Text Against Real-world Applications.

    https://arxiv.org/abs/1812.05271
    """

    @staticmethod
    def build(model_wrapper, max_num_words = None, max_percent = None):
        #
        #  we propose five bug generation methods for TEXTBUGGER:
        #
        transformation = CompositeTransformation(
            [
                # (1) Insert: Insert a space into the word.
                # Generally, words are segmented by spaces in English. Therefore,
                # we can deceive classifiers by inserting spaces into words.
                WordSwapRandomCharacterInsertion(
                    random_one=True,
                    letters_to_insert=" ",
                    skip_first_char=True,
                    skip_last_char=True,
                ),
                # (2) Delete: Delete a random character of the word except for the first
                # and the last character.
                WordSwapRandomCharacterDeletion(
                    random_one=True, skip_first_char=True, skip_last_char=True
                ),
                # (3) Swap: Swap random two adjacent letters in the word but do not
                # alter the first or last letter. This is a common occurrence when
                # typing quickly and is easy to implement.
                WordSwapNeighboringCharacterSwap(
                    random_one=True, skip_first_char=True, skip_last_char=True
                ),
                # (4) Substitute-C (Sub-C): Replace characters with visually similar
                # characters (e.g., replacing “o” with “0”, “l” with “1”, “a” with “@”)
                # or adjacent characters in the keyboard (e.g., replacing “m” with “n”).
                WordSwapHomoglyphSwap(),
                # (5) Substitute-W
                # (Sub-W): Replace a word with its topk nearest neighbors in a
                # context-aware word vector space. Specifically, we use the pre-trained
                # GloVe model [30] provided by Stanford for word embedding and set
                # topk = 5 in the experiment.
                WordSwapEmbedding(max_candidates=5),
            ]
        )

        constraints = [RepeatModification(), StopwordModification()]
        # In our experiment, we first use the Universal Sentence
        # Encoder [7], a model trained on a number of natural language
        # prediction tasks that require modeling the meaning of word
        # sequences, to encode sentences into high dimensional vectors.
        # Then, we use the cosine similarity to measure the semantic
        # similarity between original texts and adversarial texts.
        # ... "Furthermore, the semantic similarity threshold \eps is set
        # as 0.8 to guarantee a good trade-off between quality and
        # strength of the generated adversarial text."
        constraints.append(UniversalSentenceEncoder(threshold=0.8))

        # add max_words_perturbed
        if max_num_words or max_percent:
            constraints.append(textattack.constraints.overlap.MaxWordsPerturbed(max_num_words=max_num_words, max_percent=max_percent))

        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # Greedily swap words with "Word Importance Ranking".
        #
        search_method = GreedyWordSwapWIR(wir_method="delete")

        return Attack(goal_function, constraints, transformation, search_method)


In [6]:
"""

DeepWordBug
========================================
(Black-box Generation of Adversarial Text Sequences to Evade Deep Learning Classifiers)

"""

from textattack import Attack
from textattack.constraints.overlap import LevenshteinEditDistance
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import (
    CompositeTransformation,
    WordSwapNeighboringCharacterSwap,
    WordSwapRandomCharacterDeletion,
    WordSwapRandomCharacterInsertion,
    WordSwapRandomCharacterSubstitution,
)

class AddMaxWordDeepWordBugGao2018(textattack.attack_recipes.DeepWordBugGao2018):
    """Gao, Lanchantin, Soffa, Qi.

    Black-box Generation of Adversarial Text Sequences to Evade Deep
    Learning Classifiers.

    https://arxiv.org/abs/1801.04354
    """

    @staticmethod
    def build(model_wrapper, use_all_transformations=True, max_num_words = None, max_percent = None):
        #
        # Swap characters out from words. Choose the best of four potential transformations.
        #
        if use_all_transformations:
            # We propose four similar methods:
            transformation = CompositeTransformation(
                [
                    # (1) Swap: Swap two adjacent letters in the word.
                    WordSwapNeighboringCharacterSwap(),
                    # (2) Substitution: Substitute a letter in the word with a random letter.
                    WordSwapRandomCharacterSubstitution(),
                    # (3) Deletion: Delete a random letter from the word.
                    WordSwapRandomCharacterDeletion(),
                    # (4) Insertion: Insert a random letter in the word.
                    WordSwapRandomCharacterInsertion(),
                ]
            )
        else:
            # We use the Combined Score and the Substitution Transformer to generate
            # adversarial samples, with the maximum edit distance difference of 30
            # (ϵ = 30).
            transformation = WordSwapRandomCharacterSubstitution()
        #
        # Don't modify the same word twice or stopwords
        #
        constraints = [RepeatModification(), StopwordModification()]

        # add max_words_perturbed
        if max_num_words or max_percent:
            constraints.append(textattack.constraints.overlap.MaxWordsPerturbed(max_num_words=max_num_words, max_percent=max_percent))


        #
        # In these experiments, we hold the maximum difference
        # on edit distance (ϵ) to a constant 30 for each sample.
        #
        constraints.append(LevenshteinEditDistance(30))
        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # Greedily swap words with "Word Importance Ranking".
        #
        search_method = GreedyWordSwapWIR()

        return Attack(goal_function, constraints, transformation, search_method)


# Run Attack

In [ ]:
import torch
torch.cuda.set_device(2)
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/sentiment/"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-toxic"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion" 
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data" 
random_seed = 42
result_base_dir = "/data/rhhuang/notebooks/AdvTradition/toxic"
query_budget = None
label2values = {'positive': 1, 'negative': 0}
label2values = {'illicit': 1, 'benign': 0}
label2values = {'toxic': 1, 'benign': 0}

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

attack = textattack.attack_recipes.TextFoolerJin2019.build(model_wrapper)
attack_name = "TextFooler"

attack = textattack.attack_recipes.TextBuggerLi2018.build(model_wrapper)
attack_name = "TextBugger"

attack = textattack.attack_recipes.DeepWordBugGao2018.build(model_wrapper)
attack_name = "DeepWordBug"

run_adv_attack_experiment(attack,
    attack_name=attack_name,
    dataset_path=dataset_path, 
    random_seed=random_seed, 
    result_base_dir=result_base_dir,
    query_budget=query_budget,
    label2values=label2values,
    pos_label='toxic'
)

In [ ]:
model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/sentiment/"
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-toxic"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion" 
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data" 
dataset_path = "/data/ningyuanhe/EvasionFromICL/data/sentiment/" 
random_seed = 42
result_base_dir = "/data/rhhuang/notebooks/AdvTradition/toxic"
query_budget = None
label2values = {'positive': 1, 'negative': 0}
# label2values = {'illicit': 1, 'benign': 0}
# label2values = {'toxic': 1, 'benign': 0}

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

attack = textattack.attack_recipes.TextFoolerJin2019.build(model_wrapper)

run_adv_attack_experiment(attack,
    dataset_path=dataset_path, 
    random_seed=random_seed, 
    result_base_dir=result_base_dir,
    query_budget=query_budget,
    label2values=label2values,
    pos_label='negative'
)

In [ ]:
import torch
torch.cuda.set_device(2)
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/sentiment/"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-toxic"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion" 
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data" 
random_seed = 42
result_base_dir = "/data/rhhuang/notebooks/AdvTradition/toxic"
query_budget = None
label2values = {'positive': 1, 'negative': 0}
label2values = {'illicit': 1, 'benign': 0}
label2values = {'toxic': 1, 'benign': 0}

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

for word in [1, 2, 4, 8, 16]:
    attack = AddMaxWordTextFoolerJin2019.build(model_wrapper, max_num_words=word)
    attack_name = "TextFooler"

    # attack = textattack.attack_recipes.TextBuggerLi2018.build(model_wrapper)
    # attack_name = "TextBugger"

    # attack = textattack.attack_recipes.DeepWordBugGao2018.build(model_wrapper)
    # attack_name = "DeepWordBug"

    run_adv_attack_experiment(attack,
        attack_name=attack_name,
        dataset_path=dataset_path, 
        random_seed=random_seed, 
        result_base_dir=result_base_dir,
        query_budget=query_budget,
        label2values=label2values,
        pos_label='toxic'
    )

# Run Attack V2

In [ ]:
import textattack
import transformers

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

attack = textattack.attack_recipes.TextBuggerLi2018.build(model_wrapper)
    # attack_name = "TextBugger"

attack = textattack.attack_recipes.DeepWordBugGao2018.build(model_wrapper)

In [ ]:
import torch
torch.cuda.set_device(2)
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/sentiment/"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-toxic-batch32-lr5e-05-epochs3/"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion" 
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data" 
random_seed = 42
result_base_dir = "/data/rhhuang/notebooks/AdvTradition/toxic"
query_budget = None
label2values = {'positive': 1, 'negative': 0}
label2values = {'illicit': 1, 'benign': 0}
label2values = {'toxic': 1, 'benign': 0}

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

for word in [1, 2, 4, 8, 16]:
    attack = AddMaxWordTextFoolerJin2019.build(model_wrapper, max_num_words=word)
    attack_name = "TextFooler"

    attack = AddMaxWordTextBuggerLi2018.build(model_wrapper, max_num_words=word)
    attack_name = "TextBugger"

    attack = AddMaxWordDeepWordBugGao2018.build(model_wrapper, max_num_words=word)
    attack_name = "DeepWordBug"

    run_adv_attack_experiment(attack,
        attack_name=attack_name,
        dataset_path=dataset_path, 
        random_seed=random_seed, 
        result_base_dir=result_base_dir,
        query_budget=query_budget,
        label2values=label2values,
  ``      pos_label='toxic'
    )

# Illicit Promotion - Chinese

In [ ]:
import string
import textattack
from textattack import Attack
import transformers
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.pre_transformation import (
    InputColumnModification,
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapEmbedding
from textattack.transformations import CompositeTransformation
from textattack.transformations import ChineseWordSwapMaskedLM
from textattack.transformations import ChineseMorphonymCharacterSwap
from textattack.transformations import ChineseWordSwapHowNet
from textattack.transformations import ChineseHomophoneCharacterSwap
from textattack.constraints.semantics.sentence_encoders import (
    MultilingualUniversalSentenceEncoder,
)


class ChineseTextFoolerJin2019(textattack.attack_recipes.TextFoolerJin2019):
    """Jin, D., Jin, Z., Zhou, J.T., & Szolovits, P. (2019).

    Is BERT Really Robust? Natural Language Attack on Text
    Classification and Entailment.

    https://arxiv.org/abs/1907.11932
    """

    @staticmethod
    def build(model_wrapper):
        #
        # Swap words with their 50 closest embedding nearest-neighbors.
        # Embedding: Counter-fitted PARAGRAM-SL999 vectors.
        #
        transformation = WordSwapEmbedding(max_candidates=50)
        #
        # Don't modify the same word twice or the stopwords defined
        # in the TextFooler public implementation.
        #
        # fmt: off
        stopwords = set(
            ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
        )
        # stopwords = set()
        transformation = ChineseMorphonymCharacterSwap()

        # constraint
        stopwords = stopwords | set(
                [
                    "、",
                    "。",
                    "〈",
                    "〉",
                    "《",
                    "》",
                    "一",
                    "一个",
                    "一些",
                    "一何",
                    "一切",
                    "一则",
                    "一方面",
                    "一旦",
                    "一来",
                    "一样",
                    "一种",
                    "一般",
                    "一转眼",
                    "七",
                    "万一",
                    "三",
                    "上",
                    "上下",
                    "下",
                    "不",
                    "不仅",
                    "不但",
                    "不光",
                    "不单",
                    "不只",
                    "不外乎",
                    "不如",
                    "不妨",
                    "不尽",
                    "不尽然",
                    "不得",
                    "不怕",
                    "不惟",
                    "不成",
                    "不拘",
                    "不料",
                    "不是",
                    "不比",
                    "不然",
                    "不特",
                    "不独",
                    "不管",
                    "不至于",
                    "不若",
                    "不论",
                    "不过",
                    "不问",
                    "与",
                    "与其",
                    "与其说",
                    "与否",
                    "与此同时",
                    "且",
                    "且不说",
                    "且说",
                    "两者",
                    "个",
                    "个别",
                    "中",
                    "临",
                    "为",
                    "为了",
                    "为什么",
                    "为何",
                    "为止",
                    "为此",
                    "为着",
                    "乃",
                    "乃至",
                    "乃至于",
                    "么",
                    "之",
                    "之一",
                    "之所以",
                    "之类",
                    "乌乎",
                    "乎",
                    "乘",
                    "九",
                    "也",
                    "也好",
                    "也罢",
                    "了",
                    "二",
                    "二来",
                    "于",
                    "于是",
                    "于是乎",
                    "云云",
                    "云尔",
                    "五",
                    "些",
                    "亦",
                    "人",
                    "人们",
                    "人家",
                    "什",
                    "什么",
                    "什么样",
                    "今",
                    "介于",
                    "仍",
                    "仍旧",
                    "从",
                    "从此",
                    "从而",
                    "他",
                    "他人",
                    "他们",
                    "他们们",
                    "以",
                    "以上",
                    "以为",
                    "以便",
                    "以免",
                    "以及",
                    "以故",
                    "以期",
                    "以来",
                    "以至",
                    "以至于",
                    "以致",
                    "们",
                    "任",
                    "任何",
                    "任凭",
                    "会",
                    "似的",
                    "但",
                    "但凡",
                    "但是",
                    "何",
                    "何以",
                    "何况",
                    "何处",
                    "何时",
                    "余外",
                    "作为",
                    "你",
                    "你们",
                    "使",
                    "使得",
                    "例如",
                    "依",
                    "依据",
                    "依照",
                    "便于",
                    "俺",
                    "俺们",
                    "倘",
                    "倘使",
                    "倘或",
                    "倘然",
                    "倘若",
                    "借",
                    "借傥然",
                    "假使",
                    "假如",
                    "假若",
                    "做",
                    "像",
                    "儿",
                    "先不先",
                    "光",
                    "光是",
                    "全体",
                    "全部",
                    "八",
                    "六",
                    "兮",
                    "共",
                    "关于",
                    "关于具体地说",
                    "其",
                    "其一",
                    "其中",
                    "其二",
                    "其他",
                    "其余",
                    "其它",
                    "其次",
                    "具体地说",
                    "具体说来",
                    "兼之",
                    "内",
                    "再",
                    "再其次",
                    "再则",
                    "再有",
                    "再者",
                    "再者说",
                    "再说",
                    "冒",
                    "冲",
                    "况且",
                    "几",
                    "几时",
                    "凡",
                    "凡是",
                    "凭",
                    "凭借",
                    "出于",
                    "出来",
                    "分",
                    "分别",
                    "则",
                    "则甚",
                    "别",
                    "别人",
                    "别处",
                    "别是",
                    "别的",
                    "别管",
                    "别说",
                    "到",
                    "前后",
                    "前此",
                    "前者",
                    "加之",
                    "加以",
                    "区",
                    "即",
                    "即令",
                    "即使",
                    "即便",
                    "即如",
                    "即或",
                    "即若",
                    "却",
                    "去",
                    "又",
                    "又及",
                    "及",
                    "及其",
                    "及至",
                    "反之",
                    "反而",
                    "反过来",
                    "反过来说",
                    "受到",
                    "另",
                    "另一方面",
                    "另外",
                    "另悉",
                    "只",
                    "只当",
                    "只怕",
                    "只是",
                    "只有",
                    "只消",
                    "只要",
                    "只限",
                    "叫",
                    "叮咚",
                    "可",
                    "可以",
                    "可是",
                    "可见",
                    "各",
                    "各个",
                    "各位",
                    "各种",
                    "各自",
                    "同",
                    "同时",
                    "后",
                    "后者",
                    "向",
                    "向使",
                    "向着",
                    "吓",
                    "吗",
                    "否则",
                    "吧",
                    "吧哒",
                    "含",
                    "吱",
                    "呀",
                    "呃",
                    "呕",
                    "呗",
                    "呜",
                    "呜呼",
                    "呢",
                    "呵",
                    "呵呵",
                    "呸",
                    "呼哧",
                    "咋",
                    "和",
                    "咚",
                    "咦",
                    "咧",
                    "咱",
                    "咱们",
                    "咳",
                    "哇",
                    "哈",
                    "哈哈",
                    "哉",
                    "哎",
                    "哎呀",
                    "哎哟",
                    "哗",
                    "哟",
                    "哦",
                    "哩",
                    "哪",
                    "哪个",
                    "哪些",
                    "哪儿",
                    "哪天",
                    "哪年",
                    "哪怕",
                    "哪样",
                    "哪边",
                    "哪里",
                    "哼",
                    "哼唷",
                    "唉",
                    "唯有",
                    "啊",
                    "啐",
                    "啥",
                    "啦",
                    "啪达",
                    "啷当",
                    "喂",
                    "喏",
                    "喔唷",
                    "喽",
                    "嗡",
                    "嗡嗡",
                    "嗬",
                    "嗯",
                    "嗳",
                    "嘎",
                    "嘎登",
                    "嘘",
                    "嘛",
                    "嘻",
                    "嘿",
                    "嘿嘿",
                    "四",
                    "因",
                    "因为",
                    "因了",
                    "因此",
                    "因着",
                    "因而",
                    "固然",
                    "在",
                    "在下",
                    "在于",
                    "地",
                    "基于",
                    "处在",
                    "多",
                    "多么",
                    "多少",
                    "大",
                    "大家",
                    "她",
                    "她们",
                    "好",
                    "如",
                    "如上",
                    "如上所述",
                    "如下",
                    "如何",
                    "如其",
                    "如同",
                    "如是",
                    "如果",
                    "如此",
                    "如若",
                    "始而",
                    "孰料",
                    "孰知",
                    "宁",
                    "宁可",
                    "宁愿",
                    "宁肯",
                    "它",
                    "它们",
                    "对",
                    "对于",
                    "对待",
                    "对方",
                    "对比",
                    "将",
                    "小",
                    "尔",
                    "尔后",
                    "尔尔",
                    "尚且",
                    "就",
                    "就是",
                    "就是了",
                    "就是说",
                    "就算",
                    "就要",
                    "尽",
                    "尽管",
                    "尽管如此",
                    "岂但",
                    "己",
                    "已",
                    "已矣",
                    "巴",
                    "巴巴",
                    "年",
                    "并",
                    "并且",
                    "庶乎",
                    "庶几",
                    "开外",
                    "开始",
                    "归",
                    "归齐",
                    "当",
                    "当地",
                    "当然",
                    "当着",
                    "彼",
                    "彼时",
                    "彼此",
                    "往",
                    "待",
                    "很",
                    "得",
                    "得了",
                    "怎",
                    "怎么",
                    "怎么办",
                    "怎么样",
                    "怎奈",
                    "怎样",
                    "总之",
                    "总的来看",
                    "总的来说",
                    "总的说来",
                    "总而言之",
                    "恰恰相反",
                    "您",
                    "惟其",
                    "慢说",
                    "我",
                    "我们",
                    "或",
                    "或则",
                    "或是",
                    "或曰",
                    "或者",
                    "截至",
                    "所",
                    "所以",
                    "所在",
                    "所幸",
                    "所有",
                    "才",
                    "才能",
                    "打",
                    "打从",
                    "把",
                    "抑或",
                    "拿",
                    "按",
                    "按照",
                    "换句话说",
                    "换言之",
                    "据",
                    "据此",
                    "接着",
                    "故",
                    "故此",
                    "故而",
                    "旁人",
                    "无",
                    "无宁",
                    "无论",
                    "既",
                    "既往",
                    "既是",
                    "既然",
                    "日",
                    "时",
                    "时候",
                    "是",
                    "是以",
                    "是的",
                    "更",
                    "曾",
                    "替",
                    "替代",
                    "最",
                    "月",
                    "有",
                    "有些",
                    "有关",
                    "有及",
                    "有时",
                    "有的",
                    "望",
                    "朝",
                    "朝着",
                    "本",
                    "本人",
                    "本地",
                    "本着",
                    "本身",
                    "来",
                    "来着",
                    "来自",
                    "来说",
                    "极了",
                    "果然",
                    "果真",
                    "某",
                    "某个",
                    "某些",
                    "某某",
                    "根据",
                    "欤",
                    "正值",
                    "正如",
                    "正巧",
                    "正是",
                    "此",
                    "此地",
                    "此处",
                    "此外",
                    "此时",
                    "此次",
                    "此间",
                    "毋宁",
                    "每",
                    "每当",
                    "比",
                    "比及",
                    "比如",
                    "比方",
                    "没奈何",
                    "沿",
                    "沿着",
                    "漫说",
                    "点",
                    "焉",
                    "然则",
                    "然后",
                    "然而",
                    "照",
                    "照着",
                    "犹且",
                    "犹自",
                    "甚且",
                    "甚么",
                    "甚或",
                    "甚而",
                    "甚至",
                    "甚至于",
                    "用",
                    "用来",
                    "由",
                    "由于",
                    "由是",
                    "由此",
                    "由此可见",
                    "的",
                    "的确",
                    "的话",
                    "直到",
                    "相对而言",
                    "省得",
                    "看",
                    "眨眼",
                    "着",
                    "着呢",
                    "矣",
                    "矣乎",
                    "矣哉",
                    "离",
                    "秒",
                    "称",
                    "竟而",
                    "第",
                    "等",
                    "等到",
                    "等等",
                    "简言之",
                    "管",
                    "类如",
                    "紧接着",
                    "纵",
                    "纵令",
                    "纵使",
                    "纵然",
                    "经",
                    "经过",
                    "结果",
                    "给",
                    "继之",
                    "继后",
                    "继而",
                    "综上所述",
                    "罢了",
                    "者",
                    "而",
                    "而且",
                    "而况",
                    "而后",
                    "而外",
                    "而已",
                    "而是",
                    "而言",
                    "能",
                    "能否",
                    "腾",
                    "自",
                    "自个儿",
                    "自从",
                    "自各儿",
                    "自后",
                    "自家",
                    "自己",
                    "自打",
                    "自身",
                    "至",
                    "至于",
                    "至今",
                    "至若",
                    "致",
                    "般的",
                    "若",
                    "若夫",
                    "若是",
                    "若果",
                    "若非",
                    "莫不然",
                    "莫如",
                    "莫若",
                    "虽",
                    "虽则",
                    "虽然",
                    "虽说",
                    "被",
                    "要",
                    "要不",
                    "要不是",
                    "要不然",
                    "要么",
                    "要是",
                    "譬喻",
                    "譬如",
                    "让",
                    "许多",
                    "论",
                    "设使",
                    "设或",
                    "设若",
                    "诚如",
                    "诚然",
                    "该",
                    "说",
                    "说来",
                    "请",
                    "诸",
                    "诸位",
                    "诸如",
                    "谁",
                    "谁人",
                    "谁料",
                    "谁知",
                    "贼死",
                    "赖以",
                    "赶",
                    "起",
                    "起见",
                    "趁",
                    "趁着",
                    "越是",
                    "距",
                    "跟",
                    "较",
                    "较之",
                    "边",
                    "过",
                    "还",
                    "还是",
                    "还有",
                    "还要",
                    "这",
                    "这一来",
                    "这个",
                    "这么",
                    "这么些",
                    "这么样",
                    "这么点儿",
                    "这些",
                    "这会儿",
                    "这儿",
                    "这就是说",
                    "这时",
                    "这样",
                    "这次",
                    "这般",
                    "这边",
                    "这里",
                    "进而",
                    "连",
                    "连同",
                    "逐步",
                    "通过",
                    "遵循",
                    "遵照",
                    "那",
                    "那个",
                    "那么",
                    "那么些",
                    "那么样",
                    "那些",
                    "那会儿",
                    "那儿",
                    "那时",
                    "那样",
                    "那般",
                    "那边",
                    "那里",
                    "都",
                    "鄙人",
                    "鉴于",
                    "针对",
                    "阿",
                    "除",
                    "除了",
                    "除外",
                    "除开",
                    "除此之外",
                    "除非",
                    "随",
                    "随后",
                    "随时",
                    "随着",
                    "难道说",
                    "零",
                    "非",
                    "非但",
                    "非徒",
                    "非特",
                    "非独",
                    "靠",
                    "顺",
                    "顺着",
                    "首先",
                    "︿",
                    "！",
                    "＃",
                    "＄",
                    "％",
                    "＆",
                    "（",
                    "）",
                    "＊",
                    "＋",
                    "，",
                    "０",
                    "１",
                    "２",
                    "３",
                    "４",
                    "５",
                    "６",
                    "７",
                    "８",
                    "９",
                    "：",
                    "；",
                    "＜",
                    "＞",
                    "？",
                    "＠",
                    "［",
                    "］",
                    "｛",
                    "｜",
                    "｝",
                    "～",
                    "￥",
                ]
            )
        stopwords = stopwords.union(set(string.punctuation))
        # fmt: on
        constraints = [RepeatModification(), StopwordModification(stopwords=stopwords)]
        #
        # During entailment, we should only edit the hypothesis - keep the premise
        # the same.
        #
        input_column_modification = InputColumnModification(
            ["premise", "hypothesis"], {"premise"}
        )
        constraints.append(input_column_modification)
        # Minimum word embedding cosine similarity of 0.5.
        # (The paper claims 0.7, but analysis of the released code and some empirical
        # results show that it's 0.5.)
        #
        constraints.append(WordEmbeddingDistance(min_cos_sim=0.5))
        #
        # Only replace words with the same part of speech (or nouns with verbs)
        #
        constraints.append(PartOfSpeech(allow_verb_noun_swap=True))
        #
        # Universal Sentence Encoder with a minimum angular similarity of ε = 0.5.
        #
        # In the TextFooler code, they forget to divide the angle between the two
        # embeddings by pi. So if the original threshold was that 1 - sim >= 0.5, the
        # new threshold is 1 - (0.5) / pi = 0.840845057
        #
        # use_constraint = UniversalSentenceEncoder(
        #     threshold=0.840845057,
        #     metric="angular",
        #     compare_against_original=False,
        #     window_size=15,
        #     skip_text_shorter_than_window=True,
        # )
        use_constraint = MultilingualUniversalSentenceEncoder(
            threshold=0.840845057,
            metric="angular",
            compare_against_original=False,
            window_size=15,
            skip_text_shorter_than_window=True,
        )
        # 加上这个就报错
        constraints.append(use_constraint)
        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # Greedily swap words with "Word Importance Ranking".
        #
        search_method = GreedyWordSwapWIR(wir_method="delete")
        search_method = GreedyWordSwapWIR(wir_method="weighted-saliency")

        return Attack(goal_function, constraints, transformation, search_method)


In [ ]:
import torch
# torch.cuda.set_device(2)
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/sentiment/"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
model_path = "/data/rhhuang/models/bert/bert-base-multilingual-uncased-illicit_promotion-batch32-lr5e-05-epochs4/"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion" 
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data" 
random_seed = 42
result_base_dir = "/data/rhhuang/notebooks/AdvTradition/illicit_promotion/"
# result_base_dir = "/data/rhhuang/notebooks/AdvTradition/toxic"
query_budget = None
label2values = {'positive': 1, 'negative': 0}
label2values = {'illicit': 0, 'benign': 1}

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

# attack = ChineseTextFoolerJin2019.build(model_wrapper)
# attack_name = "TextFooler"


# run_adv_attack_experiment(attack,
#     attack_name=attack_name,
#     dataset_path=dataset_path, 
#     random_seed=random_seed, 
#     result_base_dir=result_base_dir,
#     query_budget=query_budget,
#     label2values=label2values,
#     pos_label='illicit',
#     max_num_words=word
# )
# 
for word in [1, 2, 4, 8, 16]:
    
    
    # attack = AddMaxWordTextFoolerJin2019.build(model_wrapper, max_num_words=word)
    # attack_name = "TextFooler"


    # run_adv_attack_experiment(attack,
    #     attack_name=attack_name,
    #     dataset_path=dataset_path, 
    #     random_seed=random_seed, 
    #     result_base_dir=result_base_dir,
    #     query_budget=query_budget,
    #     label2values=label2values,
    #     pos_label='illicit',
    #     max_num_words=word
    # )

    # attack = AddMaxWordTextBuggerLi2018.build(model_wrapper, max_num_words=word)
    # attack_name = "TextBugger"
    # run_adv_attack_experiment(attack,
    #     attack_name=attack_name,
    #     dataset_path=dataset_path, 
    #     random_seed=random_seed, 
    #     result_base_dir=result_base_dir,
    #     query_budget=query_budget,
    #     label2values=label2values,
    #     pos_label='illicit',
    #     max_num_words=word
    # )

    attack = AddMaxWordDeepWordBugGao2018.build(model_wrapper, max_num_words=word)
    attack_name = "DeepWordBug"
    run_adv_attack_experiment(attack,
        attack_name=attack_name,
        dataset_path=dataset_path, 
        random_seed=random_seed, 
        result_base_dir=result_base_dir,
        query_budget=query_budget,
        label2values=label2values,
        pos_label='illicit',
        max_num_words=word
    )

In [3]:
import torch
# torch.cuda.set_device(2)
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/sentiment/"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
model_path = "/data/rhhuang/models/bert/bert-base-multilingual-uncased-illicit_promotion-batch32-lr5e-05-epochs4/"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion" 
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data" 
random_seed = 42
result_base_dir = "/data/rhhuang/notebooks/AdvTradition/illicit_promotion/new"
# result_base_dir = "/data/rhhuang/notebooks/AdvTradition/toxic"
query_budget = None
label2values = {'positive': 1, 'negative': 0}
label2values = {'illicit': 0, 'benign': 1}

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

    







In [ ]:
attack = AddMaxWordTextFoolerJin2019.build(model_wrapper, max_num_words=16)
attack_name = "TextFooler"
attack = AddMaxWordDeepWordBugGao2018.build(model_wrapper, max_num_words=16)
attack_name = "DeepWordBug"
attack = AddMaxWordTextBuggerLi2018.build(model_wrapper, max_num_words=16)
attack_name = "TextBugger"
run_adv_attack_experiment(attack,
    attack_name=attack_name,
    dataset_path=dataset_path, 
    random_seed=random_seed, 
    result_base_dir=result_base_dir,
    query_budget=query_budget,
    label2values=label2values,
    pos_label='illicit',
    max_num_words=16
)

textattack: Unknown if model of class <class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path /data/rhhuang/notebooks/AdvTradition/illicit_promotion/new/log.csv


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (2): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (3): WordSwapHomoglyphSwap
    (4): WordSwapEmbedding(
        (max_candidates):  5
        (embedding):  WordEmbedding
      )
    )
  (constraints): 
    (0): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.8
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (1): MaxWordsPerturbed(
        (max_num_words):  16
        (compare_against_original):  True
      )
    (2): RepeatModification
    (3): StopwordModification
  (is_black_box):  True
) 



  0%|          | 0/532 [00:00<?, ?it/s]

KeyboardInterrupt: 

# New Code For Attack

In [1]:
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from transformers import BertModel, BertTokenizer
import torch
from abc import ABC, abstractmethod
from collections import defaultdict
import os
import pickle

import numpy as np
import torch

from textattack.shared import utils, AbstractWordEmbedding


class MBERTWordEmbedding(AbstractWordEmbedding):
    def __init__(self, model_name="/data/rhhuang/models/bert/bert-base-multilingual-uncased-illicit_promotion-batch32-lr5e-05-epochs4/",device="cuda"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name).to(device)
        self.model.eval()
        self.device = device
        self._mse_dist_mat = defaultdict(dict)
        self._cos_sim_mat = defaultdict(dict)
        
        # 预先计算并存储所有词的嵌入向量
        self.all_embeddings = self._precompute_all_embeddings()

    def _precompute_all_embeddings(self):
        all_embeddings = []
        for word in self.tokenizer.vocab:
            inputs = self.tokenizer(word, return_tensors="pt").to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            embedding = outputs.last_hidden_state[0, 1:-1].mean(dim=0).cpu().numpy()
            all_embeddings.append(embedding)
        return torch.tensor(all_embeddings).to(self.device)

    def __getitem__(self, index):
        if isinstance(index, str):
            inputs = self.tokenizer(index, return_tensors="pt").to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            embeddings = outputs.last_hidden_state[0, 1:-1].mean(dim=0).cpu().numpy()
            return embeddings
        else:
            raise ValueError("Index must be a string (word)")

    def word2index(self, word):
        inputs = self.tokenizer(word, return_tensors="pt")
        return inputs["input_ids"].squeeze(0)[1].item()

    def index2word(self, index):
        return self.tokenizer.decode([index], skip_special_tokens=True)

    def get_mse_dist(self, a, b):
        try:
            mse_dist = self._mse_dist_mat[a][b]
        except KeyError:
            e1 = self[a]
            e2 = self[b]
            e1 = torch.tensor(e1).to(self.device)
            e2 = torch.tensor(e2).to(self.device)
            mse_dist = torch.sum((e1 - e2) ** 2).item()
            self._mse_dist_mat[a][b] = mse_dist
        return mse_dist

    def get_cos_sim(self, a, b):
        try:
            cos_sim = self._cos_sim_mat[a][b]
        except KeyError:
            e1 = self[a]
            e2 = self[b]
            e1 = torch.tensor(e1).to(self.device)
            e2 = torch.tensor(e2).to(self.device)
            cos_sim = torch.nn.CosineSimilarity(dim=0)(e1, e2).item()
            self._cos_sim_mat[a][b] = cos_sim
        return cos_sim

    def nearest_neighbours(self, index, topn):
        word = self.index2word(index)
        inputs = self.tokenizer(word, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        embeddings = outputs.last_hidden_state[0, 1:-1].mean(dim=0).unsqueeze(0)

        # 计算与所有词的欧几里得距离
        distances = torch.norm(self.all_embeddings - embeddings, dim=1)
        topk_indices = distances.topk(topn + 1, largest=False)[1][1:]  # 排除自身

        return topk_indices.cpu().numpy()
    


import textattack
import transformers
import os
import csv
import pandas as pd


def run_adv_attack_experiment(
        attack,
        attack_name,
        dataset_path="/data/rhhuang/notebooks/Adv4ICL/data/sentiment/",
        random_seed=42,
        result_base_dir="/data/rhhuang/notebooks/AdvTradition",
        query_budget=None,
        label2values={'positive': 1, 'negative': 0},
        pos_label='negative',
        max_num_words=1
    ):
    test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
    test_df = test_df[test_df['label'] == pos_label]
    test_df['label_value'] = test_df['label'].map(label2values)
    
    # 创建 text 到 label 的映射
    text_label_map = dict(zip(test_df['text'], test_df['label']))

    # 创建 text 到 idx 的映射
    text_idx_map = dict(zip(test_df['text'], test_df['idx']))

    # 提取所需数据
    test_data = [(row['text'], row['label_value']) for _, row in test_df.iterrows()]
    # 创建TextAttack的数据集
    eval_dataset = textattack.datasets.Dataset(test_data)

    os.makedirs(result_base_dir, exist_ok=True)
    # Attack 20 samples with CSV logging and checkpoint saved every 5 interval
    attack_args = textattack.AttackArgs(
        num_examples=-1,
        log_to_csv=os.path.join(result_base_dir, "log.csv"),
        checkpoint_interval=5,
        disable_stdout=True,
        query_budget=query_budget,
        random_seed=random_seed,
    )

    attacker = textattack.Attacker(attack, eval_dataset, attack_args)
    result = attacker.attack_dataset()
    
    output_file = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_resultsV2.csv")
    output_file_success = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_results_successV2.csv")
    output_file_fail = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_results_failV2.csv")
    output_file_skip = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_results_skipV2.csv")
    output_file_extra = os.path.join(result_base_dir, f"{attack_name}_max_num_words{max_num_words}_attack_extra_resultsV2.csv")
    
    # 定义文件头
    header = ["original_text", "perturbed_text", "label"]
    extra_header = ["idx", "text", "label_value", "label"]

    # 读取并保存结果
    with open(output_file_success, mode="w", newline='', encoding="utf-8") as file_success, \
         open(output_file_skip, mode="w", newline='', encoding="utf-8") as file_skip, \
         open(output_file_fail, mode="w", newline='', encoding="utf-8") as file_fail, \
         open(output_file, mode="w", newline='', encoding="utf-8") as file, \
         open(output_file_extra, mode="w", newline='', encoding="utf-8") as file_extra:

        writer = csv.writer(file)
        writer.writerow(header)

        writer_success = csv.writer(file_success)
        writer_success.writerow(header)

        writer_fail = csv.writer(file_fail)
        writer_fail.writerow(header)

        writer_skip = csv.writer(file_skip)
        writer_skip.writerow(header)

        extra_writer = csv.writer(file_extra)
        extra_writer.writerow(extra_header)

        for attack_result in result:
            original_text = attack_result.original_text()
            perturbed_text = attack_result.perturbed_text()
            label = text_label_map[original_text]
            label_value = label2values[label]
            
            if isinstance(attack_result, textattack.attack_results.SuccessfulAttackResult):
                writer_success.writerow([original_text, perturbed_text, label])
            elif isinstance(attack_result, textattack.attack_results.FailedAttackResult):
                writer_fail.writerow([original_text, perturbed_text, label])
            elif isinstance(attack_result, textattack.attack_results.SkippedAttackResult):
                writer_skip.writerow([original_text, perturbed_text, label])
            else:
                raise Exception("Unknown Type!")
            
            writer.writerow([original_text, perturbed_text, label])
            idx = text_idx_map[original_text]
            extra_writer.writerow([idx, perturbed_text, label_value, label])

    print(f"攻击结果已保存到 {output_file}")
    print(f"附加结果已保存到 {output_file_extra}")

import unicodedata
def get_all_language_letters():
    all_letters = []
    # Unicode 范围是从 0 到 0x10FFFF（Unicode 的最大码点）
    for codepoint in range(0x110000):  
        char = chr(codepoint)
        # 通过判断字符的类别是否是字母（L 开头的分类）
        if unicodedata.category(char).startswith("L"):
            all_letters.append(char)
    return all_letters


all_language_letters = get_all_language_letters()
"""

TextFooler (Is BERT Really Robust?)
===================================================
A Strong Baseline for Natural Language Attack on Text Classification and Entailment)

"""
import textattack
from textattack import Attack
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.pre_transformation import (
    InputColumnModification,
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapEmbedding

my_embedding = MBERTWordEmbedding()


2025-01-16 01:49:57.017531: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-16 01:49:57.031854: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736992197.050993 4012647 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736992197.056987 4012647 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-16 01:49:57.077535: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:

class AddMaxWordTextFoolerJin2019(textattack.attack_recipes.TextFoolerJin2019):
    """Jin, D., Jin, Z., Zhou, J.T., & Szolovits, P. (2019).

    Is BERT Really Robust? Natural Language Attack on Text
    Classification and Entailment.

    https://arxiv.org/abs/1907.11932
    """

    @staticmethod
    def build(model_wrapper, max_num_words = None, max_percent = None):
        #
        # Swap words with their 50 closest embedding nearest-neighbors.
        # Embedding: Counter-fitted PARAGRAM-SL999 vectors.
        #
        transformation = WordSwapEmbedding(max_candidates=50, embedding=my_embedding, letters_to_insert=all_language_letters)
        #
        # Don't modify the same word twice or the stopwords defined
        # in the TextFooler public implementation.
        #
        # fmt: off
        stopwords = set(
            ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
        )
        # fmt: on
        constraints = [RepeatModification(), StopwordModification(stopwords=stopwords)]
        #
        # During entailment, we should only edit the hypothesis - keep the premise
        # the same.
        #
        input_column_modification = InputColumnModification(
            ["premise", "hypothesis"], {"premise"}
        )
        constraints.append(input_column_modification)
        # Minimum word embedding cosine similarity of 0.5.
        # (The paper claims 0.7, but analysis of the released code and some empirical
        # results show that it's 0.5.)
        #
        constraints.append(WordEmbeddingDistance(min_cos_sim=0.5))
        #
        # Only replace words with the same part of speech (or nouns with verbs)
        #
        constraints.append(PartOfSpeech(allow_verb_noun_swap=True))
        #
        # Universal Sentence Encoder with a minimum angular similarity of ε = 0.5.
        #
        # In the TextFooler code, they forget to divide the angle between the two
        # embeddings by pi. So if the original threshold was that 1 - sim >= 0.5, the
        # new threshold is 1 - (0.5) / pi = 0.840845057
        #
        use_constraint = UniversalSentenceEncoder(
            threshold=0.840845057,
            metric="angular",
            compare_against_original=False,
            window_size=15,
            skip_text_shorter_than_window=True,
        )
        constraints.append(use_constraint)

        # add max_words_perturbed
        if max_num_words or max_percent:
            constraints.append(textattack.constraints.overlap.MaxWordsPerturbed(max_num_words=max_num_words, max_percent=max_percent))
        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # Greedily swap words with "Word Importance Ranking".
        #
        search_method = GreedyWordSwapWIR(wir_method="delete")

        return Attack(goal_function, constraints, transformation, search_method)
    



"""

TextBugger
===============

(TextBugger: Generating Adversarial Text Against Real-world Applications)

"""
import textattack
from textattack import Attack
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import (
    CompositeTransformation,
    WordSwapEmbedding,
    WordSwapHomoglyphSwap,
    WordSwapNeighboringCharacterSwap,
    WordSwapRandomCharacterDeletion,
    WordSwapRandomCharacterInsertion,
)




class AddMaxWordTextBuggerLi2018(textattack.attack_recipes.TextBuggerLi2018):
    """Li, J., Ji, S., Du, T., Li, B., and Wang, T. (2018).

    TextBugger: Generating Adversarial Text Against Real-world Applications.

    https://arxiv.org/abs/1812.05271
    """

    @staticmethod
    def build(model_wrapper, max_num_words = None, max_percent = None):
        #
        #  we propose five bug generation methods for TEXTBUGGER:
        #
        transformation = CompositeTransformation(
            [
                # (1) Insert: Insert a space into the word.
                # Generally, words are segmented by spaces in English. Therefore,
                # we can deceive classifiers by inserting spaces into words.
                WordSwapRandomCharacterInsertion(
                    random_one=True,
                    letters_to_insert=" ",
                    skip_first_char=True,
                    skip_last_char=True,
                ),
                # (2) Delete: Delete a random character of the word except for the first
                # and the last character.
                WordSwapRandomCharacterDeletion(
                    random_one=True, skip_first_char=True, skip_last_char=True,letters_to_insert=all_language_letters
                ),
                # (3) Swap: Swap random two adjacent letters in the word but do not
                # alter the first or last letter. This is a common occurrence when
                # typing quickly and is easy to implement.
                WordSwapNeighboringCharacterSwap(
                    random_one=True, skip_first_char=True, skip_last_char=True,letters_to_insert=all_language_letters
                ),
                # (4) Substitute-C (Sub-C): Replace characters with visually similar
                # characters (e.g., replacing “o” with “0”, “l” with “1”, “a” with “@”)
                # or adjacent characters in the keyboard (e.g., replacing “m” with “n”).
                WordSwapHomoglyphSwap(letters_to_insert=all_language_letters),
                # (5) Substitute-W
                # (Sub-W): Replace a word with its topk nearest neighbors in a
                # context-aware word vector space. Specifically, we use the pre-trained
                # GloVe model [30] provided by Stanford for word embedding and set
                # topk = 5 in the experiment.
                WordSwapEmbedding(max_candidates=5, embedding=my_embedding, letters_to_insert=all_language_letters),
            ]
        )

        constraints = [RepeatModification(), StopwordModification()]
        # In our experiment, we first use the Universal Sentence
        # Encoder [7], a model trained on a number of natural language
        # prediction tasks that require modeling the meaning of word
        # sequences, to encode sentences into high dimensional vectors.
        # Then, we use the cosine similarity to measure the semantic
        # similarity between original texts and adversarial texts.
        # ... "Furthermore, the semantic similarity threshold \eps is set
        # as 0.8 to guarantee a good trade-off between quality and
        # strength of the generated adversarial text."
        constraints.append(UniversalSentenceEncoder(threshold=0.8))

        # add max_words_perturbed
        if max_num_words or max_percent:
            constraints.append(textattack.constraints.overlap.MaxWordsPerturbed(max_num_words=max_num_words, max_percent=max_percent))

        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # Greedily swap words with "Word Importance Ranking".
        #
        search_method = GreedyWordSwapWIR(wir_method="delete")

        return Attack(goal_function, constraints, transformation, search_method)


"""

DeepWordBug
========================================
(Black-box Generation of Adversarial Text Sequences to Evade Deep Learning Classifiers)

"""

from textattack import Attack
from textattack.constraints.overlap import LevenshteinEditDistance
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import (
    CompositeTransformation,
    WordSwapNeighboringCharacterSwap,
    WordSwapRandomCharacterDeletion,
    WordSwapRandomCharacterInsertion,
    WordSwapRandomCharacterSubstitution,
)

class AddMaxWordDeepWordBugGao2018(textattack.attack_recipes.DeepWordBugGao2018):
    """Gao, Lanchantin, Soffa, Qi.

    Black-box Generation of Adversarial Text Sequences to Evade Deep
    Learning Classifiers.

    https://arxiv.org/abs/1801.04354
    """

    @staticmethod
    def build(model_wrapper, use_all_transformations=True, max_num_words = None, max_percent = None):
        #
        # Swap characters out from words. Choose the best of four potential transformations.
        #
        if use_all_transformations:
            # We propose four similar methods:
            transformation = CompositeTransformation(
                [
                    # (1) Swap: Swap two adjacent letters in the word.
                    WordSwapNeighboringCharacterSwap(letters_to_insert=all_language_letters),
                    # (2) Substitution: Substitute a letter in the word with a random letter.
                    WordSwapRandomCharacterSubstitution(letters_to_insert=all_language_letters),
                    # (3) Deletion: Delete a random letter from the word.
                    WordSwapRandomCharacterDeletion(letters_to_insert=all_language_letters),
                    # (4) Insertion: Insert a random letter in the word.
                    WordSwapRandomCharacterInsertion(letters_to_insert=all_language_letters),
                ]
            )
        else:
            # We use the Combined Score and the Substitution Transformer to generate
            # adversarial samples, with the maximum edit distance difference of 30
            # (ϵ = 30).
            transformation = WordSwapRandomCharacterSubstitution(letters_to_insert=all_language_letters)
        #
        # Don't modify the same word twice or stopwords
        #
        constraints = [RepeatModification(), StopwordModification()]

        # add max_words_perturbed
        if max_num_words or max_percent:
            constraints.append(textattack.constraints.overlap.MaxWordsPerturbed(max_num_words=max_num_words, max_percent=max_percent))


        #
        # In these experiments, we hold the maximum difference
        # on edit distance (ϵ) to a constant 30 for each sample.
        #
        constraints.append(LevenshteinEditDistance(30))
        #
        # Goal is untargeted classification
        #
        goal_function = UntargetedClassification(model_wrapper)
        #
        # Greedily swap words with "Word Importance Ranking".
        #
        search_method = GreedyWordSwapWIR()

        return Attack(goal_function, constraints, transformation, search_method)


In [3]:
import torch
# torch.cuda.set_device(2)
# model_path = "/data/rhhuang/models/bert/bert-base-uncased-sst2"
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/sentiment/"
model_path = "/data/rhhuang/models/bert/bert-base-uncased-ip"
model_path = "/data/rhhuang/models/bert/bert-base-multilingual-uncased-illicit_promotion-batch32-lr5e-05-epochs4/"
dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/illicit_promotion" 
# dataset_path = "/data/rhhuang/notebooks/Adv4ICL/data/toxic_text/data" 
random_seed = 42
result_base_dir = "/data/rhhuang/notebooks/AdvTradition/illicit_promotion/new2"
# result_base_dir = "/data/rhhuang/notebooks/AdvTradition/toxic"
query_budget = None
label2values = {'positive': 1, 'negative': 0}
label2values = {'illicit': 0, 'benign': 1}

model = transformers.AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_path)
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)


In [5]:
import importlib
importlib.reload(textattack)
for word in [1, 2, 4, 8, 16]:
    attack = AddMaxWordTextFoolerJin2019.build(model_wrapper, max_num_words=word)
    attack_name = "TextFooler"
    run_adv_attack_experiment(attack,
        attack_name=attack_name,
        dataset_path=dataset_path, 
        random_seed=random_seed, 
        result_base_dir=result_base_dir,
        query_budget=query_budget,
        label2values=label2values,
        pos_label='illicit',
        max_num_words=word
    )
    attack = AddMaxWordDeepWordBugGao2018.build(model_wrapper, max_num_words=word)
    attack_name = "DeepWordBug"
    run_adv_attack_experiment(attack,
        attack_name=attack_name,
        dataset_path=dataset_path, 
        random_seed=random_seed, 
        result_base_dir=result_base_dir,
        query_budget=query_budget,
        label2values=label2values,
        pos_label='illicit',
        max_num_words=word
    )
    attack = AddMaxWordTextBuggerLi2018.build(model_wrapper, max_num_words=word)
    attack_name = "TextBugger"
    run_adv_attack_experiment(attack,
        attack_name=attack_name,
        dataset_path=dataset_path, 
        random_seed=random_seed, 
        result_base_dir=result_base_dir,
        query_budget=query_budget,
        label2values=label2values,
        pos_label='illicit',
        max_num_words=word
    )

textattack: Unknown if model of class <class 'transformers.models.bert.modeling_bert.BertForSequenceClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path /data/rhhuang/notebooks/AdvTradition/illicit_promotion/new2/log.csv


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  MBERTWordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): MaxWordsPerturbed(
        (max_num_words):  1
        (compare_against_original):  True
      )
    (4): RepeatModification
    (5): StopwordModi

ValueError: substring not found

# Defense

In [ ]:
import textattack
import transformers
import pandas as pd
import os
import torch
torch.cuda.set_device(2)
textattack.shared.utils.device = os.environ.get(
    "TA_DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

model = transformers.AutoModelForSequenceClassification.from_pretrained("/data/rhhuang/models/bert/bert-base-uncased") #.to("cpu")
tokenizer = transformers.AutoTokenizer.from_pretrained("/data/rhhuang/models/bert/bert-base-uncased")
model_wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

# We only use DeepWordBugGao2018 to demonstration purposes.
attack = textattack.attack_recipes.DeepWordBugGao2018.build(model_wrapper)
# TODO: load our datasets
test_df = pd.read_csv("/data/rhhuang/notebooks/Adv4ICL/data/sentiment/test.csv")
test_df['label_value'] = test_df['label'].map({'positive': 1, 'negative': 0})
# 提取所需数据
test_data = [(row['text'], row['label_value']) for _, row in test_df.iterrows()]
# 创建TextAttack的数据集
eval_dataset = textattack.datasets.Dataset(test_data)
eval_dataset.input_columns = tuple(eval_dataset.input_columns)
train_df = pd.read_csv("/data/rhhuang/notebooks/Adv4ICL/data/sentiment/train.csv")
train_df['label_value'] = train_df['label'].map({'positive': 1, 'negative': 0})
# 提取所需数据
train_data = [(row['text'], row['label_value']) for _, row in train_df.iterrows()]

# 创建TextAttack的数据集
train_dataset = textattack.datasets.Dataset(train_data)
train_dataset.input_columns = tuple(train_dataset.input_columns)
# Train for 3 epochs with 1 initial clean epochs, 1000 adversarial examples per epoch, learning rate of 5e-5, and effective batch size of 32 (8x4).
training_args = textattack.TrainingArgs(
        num_epochs=3,
    num_clean_epochs=1,
    num_train_adv_examples=1000,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    log_to_tb=True,
)

trainer = textattack.Trainer(
        model_wrapper,
    "classification",
    attack,
    train_dataset,
    eval_dataset,
    training_args,
)
trainer.train()

In [ ]:
tuple(train_dataset.input_columns)+ ("_example_type",),

# Augment

In [ ]:
# import transformations, contraints, and the Augmenter
from textattack.transformations import WordSwapRandomCharacterDeletion
from textattack.transformations import WordSwapQWERTY
from textattack.transformations import CompositeTransformation

from textattack.constraints.pre_transformation import RepeatModification
from textattack.constraints.pre_transformation import StopwordModification

from textattack.augmentation import Augmenter
from textattack import Attack
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.pre_transformation import (
    InputColumnModification,
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import WordSwapEmbedding



In [ ]:
transformation = WordSwapEmbedding(max_candidates=50)
#
# Don't modify the same word twice or the stopwords defined
# in the TextFooler public implementation.
#
# fmt: off
stopwords = set(
    ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
)
# fmt: on
constraints = [RepeatModification(), StopwordModification(stopwords=stopwords)]
#
# During entailment, we should only edit the hypothesis - keep the premise
# the same.
#
input_column_modification = InputColumnModification(
    ["premise", "hypothesis"], {"premise"}
)
constraints.append(input_column_modification)
# Minimum word embedding cosine similarity of 0.5.
# (The paper claims 0.7, but analysis of the released code and some empirical
# results show that it's 0.5.)
#
constraints.append(WordEmbeddingDistance(min_cos_sim=0.5))
#
# Only replace words with the same part of speech (or nouns with verbs)
#
constraints.append(PartOfSpeech(allow_verb_noun_swap=True))
#
# Universal Sentence Encoder with a minimum angular similarity of ε = 0.5.
#
# In the TextFooler code, they forget to divide the angle between the two
# embeddings by pi. So if the original threshold was that 1 - sim >= 0.5, the
# new threshold is 1 - (0.5) / pi = 0.840845057
#
use_constraint = UniversalSentenceEncoder(
    threshold=0.840845057,
    metric="angular",
    compare_against_original=False,
    window_size=15,
    skip_text_shorter_than_window=True,
)
constraints.append(use_constraint)
#
# Goal is untargeted classification
#
goal_function = UntargetedClassification(model_wrapper)
#
# Greedily swap words with "Word Importance Ranking".
#
search_method = GreedyWordSwapWIR(wir_method="delete")

In [ ]:
# Set up transformation using CompositeTransformation()
# transformation = CompositeTransformation(
#     [WordSwapRandomCharacterDeletion(), WordSwapQWERTY()]
# )
# # Set up constraints
# constraints = [RepeatModification(), StopwordModification()]
# Create augmenter with specified parameters
augmenter = Augmenter(
    transformation=transformation,
    constraints=constraints,
    pct_words_to_swap=0.5,
    transformations_per_example=10,
)
s = "What I cannot create, I do not understand."
# Augment!
augmenter.augment(s)